In [2]:
import os 
import sys
from google import genai 
from google.cloud import bigquery 
import vertexai
import numpy as np
import math 

from search_eval_utils import access_secret_version

In [3]:
current_dir = os.getcwd() 
project_root = os.path.abspath(os.path.join(current_dir, '../../..'))
cloud_function_root = os.path.abspath(os.path.join(project_root, 'CloudFunction')) 
utils_root = os.path.abspath(os.path.join(project_root, 'CloudFunction/utils'))

if project_root not in sys.path: 
    sys.path.insert(0, project_root) 
if cloud_function_root not in sys.path:
    sys.path.insert(0, cloud_function_root)
if utils_root not in sys.path:
    sys.path.insert(0, utils_root)

In [4]:
from send_relevent_project_to_queue_utils import haversine_distance_miles_numpy

In [5]:
PROJECT_ID_DEV = 'proj-sales-recommender-dev'
PROJECT_ID_PROD = 'proj-sales-recommender-prod'

LOCATION = 'us-central1'
DATASET_DEV = 'sales_recommender_dev'
DATASET_PROD = 'sales_recommender_prod'

CC_TABLE = 'construct_connect_feed'
DODGE_TABLE = "dodge_feed"  

# Gemini vars 
SECRET_NAME = "gemini_api_key"
MODEL_ID = "gemini-2.0-flash-001"

In [6]:
# bq_client = bigquery.Client()
# vertexai.init(project=PROJECT_ID, location=LOCATION)

# # Init Gemini client 
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/Kaylee.Rosendahl/fbmSalesRecommender/sa-key.json"
os.environ["GEMINI_API_KEY"] = access_secret_version(PROJECT_ID_DEV, SECRET_NAME)
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [7]:
from google.oauth2 import service_account
dev_credentials = service_account.Credentials.from_service_account_file(
    "/Users/Kaylee.Rosendahl/fbmSalesRecommender/sa-key.json"
)
prod_credentials = service_account.Credentials.from_service_account_file(
    "/Users/Kaylee.Rosendahl/fbmSalesRecommender/prod-sa-key.json"
)

In [8]:
bq_client_dev = bigquery.Client(credentials=dev_credentials, project=PROJECT_ID_DEV)
bq_client_prod = bigquery.Client(credentials=prod_credentials, project=PROJECT_ID_PROD)

In [9]:
def get_df_from_bq(client, query): 
    query_job = client.query(query)
    return query_job.to_dataframe()

In [10]:
from enum import Enum 
from pydantic import BaseModel
from typing import Type, TypeVar 

class Match(Enum): 
    MATCH = "match"
    NO_MATCH = "no_match"
    UNSURE = "unsure"

class ProjectDuplicateResult(BaseModel): 
    project_match: Match 
    Reasoning: str

In [11]:
T = TypeVar('T')

def run_prompt_duplicate_projects(gemini_client, model_id: str, 
                                    project_1_json: str, 
                                    project_2_json: str, 
                                    project_1_title: str, 
                                    project_2_title: str,
                                    distance: float, 
                                    project_1_owner: str = None, 
                                    project_2_owner: str = None,
                                    response_type=Type[T]):

    prompt = f"""

        Your task is to determine if two construction projects are duplicates of each other based on the provided details.

        **Instructions:**
        1. **Carefully analyze the provided JSON data** details for Project 1 and Project 2, including features, plans, and description of work.

        2. **Analyze the project titles** to understand the context and scope of each project.

        3. **Consider the distance** between the two projects, which is provided in miles. 

        4. **Consider the project owners** if that information is available.

        5. **Determine if the two projects are duplicates** or unsure based on the provided information.

        6. **Formulate reasoning** for your decision. 
        
        7. **Respond as a "match", "no match", or "unsure"** based on your analysis, along with your reasoning.

        **Project 1:** 

        **Title:** {project_1_title} 

        {"**Owner:** " + project_1_owner if project_1_owner else ""}

        **JSON Data:**
        {project_1_json}

        **Project 2:**

        **Title:** {project_2_title}

        {"**Owner:** " + project_2_owner if project_2_owner else ""}

        **JSON Data:**

        {project_2_json}

        **Distance:** {distance} miles
        
        **Example Output Format:** 
        {{
            "project_match": "match" | "no_match" | "unsure",
            "Reasoning": "Your reasoning here."
        }}

        """
        
    generation_config = { 
        "temperature": 0,
        "candidate_count": 1,
        "max_output_tokens": 2048,
        "response_mime_type": "application/json", 
        "response_schema": response_type
    }

    # Generate content using the model
    response = gemini_client.models.generate_content(
        model=model_id,
        contents=prompt, 
        config=generation_config,
    )

    if issubclass(response_type, Enum): 
        return response_type(response.text.strip())
    else: 
        return response.parsed
    

In [12]:
from fuzzywuzzy import fuzz 
import pandas as pd 
def is_same_project(project_1_data: pd.Series,
                    project_2_data: pd.Series,
                    p1_title_col: str, 
                    p2_title_col: str, 
                    p1_lat_col: str, 
                    p1_long_col: str,
                    p2_lat_col: str,
                    p2_long_col: str,
                    p1_owner: str = None, 
                    p2_owner: str = None,
                    distance_threshold_miles: float = 0.5,
                    fuzzy_threshold: float = 50) -> ProjectDuplicateResult:
    
    """
    Criteria for match: 
    - Project locations are within a certain distance threshold (default 10 miles)
    - Project titles are similar enough (default fuzzy threshold 50)
    - If both criteria are met, use LLM to confirm match, no match, or unsure

    Args: 
        project_1_data (pd.Series): Data for the first project.
        project_2_data (pd.Series): Data for the second project.
        p1_title_col (str): Column name for the title of project 1.
        p2_title_col (str): Column name for the title of project 2.
        p1_lat_col (str): Column name for latitude of project 1.
        p1_long_col (str): Column name for longitude of project 1.
        p2_lat_col (str): Column name for latitude of project 2.
        p2_long_col (str): Column name for longitude of project 2.
        distance_threshold_miles (float): Maximum distance (in miles) to consider projects as duplicates.
        fuzzy_threshold (float): Minimum fuzzy match score to consider titles as similar. Range [0, 100]

    Returns: 
        ProjectDuplicateResult: Result indicating whether the projects are a match, no match, or unsure, along with reasoning.
    """ 

    try: 
        p1_title = project_1_data[p1_title_col]
        p2_title = project_2_data[p2_title_col] 

        p1_coords = (
            float(project_1_data[p1_lat_col]),
            float(project_1_data[p1_long_col])
        )

        p2_coords = (
            float(project_2_data[p2_lat_col]),
            float(project_2_data[p2_long_col]),
        )

    except KeyError as e:
        print(f"KeyError: {e}. Please check the column names in the data.")
        return 
    except ValueError as e:
        print(f"ValueError: {e}. Please check the data types of the coordinates.")
        return
    except Exception as e:
        print(f"Unexpected error: {e}. Please check the input data.")
        return 
    

    # Calculate distance between projects
    try: 
        distance = haversine_distance_miles_numpy(
            p1_coords[0],
            p1_coords[1],
            np.array([p2_coords[0]]), 
            np.array([p2_coords[1]]),
        )[0]
    except Exception as e:
        print(f"Error calculating distance: {e}")
        return
    
    # If distance exceeds threshold, return no match
    if distance > distance_threshold_miles:
        return distance, ProjectDuplicateResult(
            project_match=Match.NO_MATCH, 
            Reasoning= f"""Distance (miles) between projects exceeds threshold: 
                        distance {distance} > threshold {distance_threshold_miles}"""
        )
    
    # Normalize titles
    p1_title = str(p1_title).lower().strip()
    p2_title = str(p2_title).lower().strip()

    # Check if titles are similar enough
    match_ratio = fuzz.partial_ratio(p1_title, p2_title) # use most similar substring 

    # If titles are not similar enough, return no match
    if match_ratio < fuzzy_threshold:
        return distance, ProjectDuplicateResult(
            project_match=Match.NO_MATCH, 
            Reasoning=f"""Titles '{p1_title}' and '{p2_title}' are not similar enough: 
                        match ratio {match_ratio} < threshold {fuzzy_threshold}"""
        )
    
    project_1_json = project_1_data.drop(
        labels=[p1_title_col, p1_long_col, p1_lat_col]
    ).to_json()

    project_2_json = project_2_data.drop(
        labels=[p2_title_col, p2_long_col, p2_lat_col]
    ).to_json()
    
    llm_match_response = run_prompt_duplicate_projects(
        gemini_client, MODEL_ID, 
        project_1_json=project_1_json, 
        project_2_json=project_2_json,
        project_1_title=p1_title, 
        project_2_title=p2_title, 
        distance=distance,
        project_1_owner=p1_owner,
        project_2_owner=p2_owner,
        response_type=ProjectDuplicateResult
    )

    return distance, llm_match_response


/Users/Kaylee.Rosendahl/.pyenv/versions/3.12.0/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [13]:
# Extracts owner from ConstructConnect project JSON
from typing import Optional

def fetch_owner_from_cc(project_json: dict) -> Optional[str]:
    """ 
    "Companies" field is expected to be a list. 
    The first item in this list expected to be a dictionary containing "Company" list. 
    Searches for a company with the role of "owner" and returns its name if found.
    """

    companies_data = project_json.get("Companies")
    if not isinstance(companies_data, (list, np.ndarray)) or not companies_data:
        return None 
    
    company_group = companies_data[0] 
    if not isinstance(company_group, dict):
        return None
    
    company_list = company_group.get("Company") 
    if not isinstance(company_list, (list, np.ndarray)):
        return None
    
    for company_details in company_list: 
        if isinstance(company_details, dict): 
            role = company_details.get("Role") 
            if isinstance(role, str) and role.lower() == "owner":
                company_name = company_details.get("Name")
                if isinstance(company_name, str):
                    return company_name.lower().strip()
            
    return None

# Extracts owner from Dodge project JSON
def fetch_owner_from_dodge(project_json):
    """ 
    The "Companies" field is expected to be a dictionary. 
    The "Company" field within this dictionary is expected to be a list of companies. 
    Searches for a company with the "FactorType" of "owner" and returns its name if found.
    """

    companies_data = project_json.get("Companies") 
    if not isinstance(companies_data, dict):
        return None 
    
    companies_list = companies_data.get("Company") 
    if not isinstance(companies_list, (list, np.ndarray)):
        return None
    
    for company_details in companies_list:
        if isinstance(company_details, dict): 
            factor_type = company_details.get("FactorType") 
            if isinstance(factor_type, str) and factor_type.lower() == "owner":
                company_name = company_details.get("CompanyName")
                if isinstance(company_name, str):
                    return company_name.lower().strip()

    return None

In [14]:
# Extracts owner from project data based on source
def fetch_owner_from_project_data(project_data: dict, source: str):
    
    # Convert pd.Series to dict if necessary
    if isinstance(project_data, pd.Series):
        project_json = project_data.to_dict()
    else: 
        project_json = project_data
        
    try: 
        if source == "construct_connect": 
            return fetch_owner_from_cc(project_json)
        elif source == "dodge": 
            return fetch_owner_from_dodge(project_json)
        
    except Exception as e:
        print(f"Error fetching owner from project data: {e}")
        return None

In [23]:
cc_projects = get_df_from_bq(
    bq_client_prod, 
    f"""SELECT *, 
    `Addresses_Address`[SAFE_OFFSET(0)].Longitude AS Longitude,
    `Addresses_Address`[SAFE_OFFSET(0)].Latitude AS Latitude,
    FROM `{PROJECT_ID_PROD}.{DATASET_PROD}.{CC_TABLE}` 
    where sourceFileCreationTime >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 7 DAY) 
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ProjectID ORDER BY sourceFileCreationTime DESC ) = 1"""
)

In [24]:
dodge_projects = get_df_from_bq(
    bq_client_dev, 
    f"""SELECT * FROM `{PROJECT_ID_DEV}.{DATASET_DEV}.{DODGE_TABLE}` 
    where sourceFileCreationTime >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 7 DAY) 
    QUALIFY ROW_NUMBER() OVER (PARTITION BY DRNumber ORDER BY sourceFileCreationTime DESC ) = 1 """
)

In [25]:
len(cc_projects), len(dodge_projects)

(28794, 20652)

In [15]:
cc_projects['Owner'] = cc_projects.apply(
    lambda row: fetch_owner_from_project_data(row, source="construct_connect"),
    axis=1
)
dodge_projects['Owner'] = dodge_projects.apply(
    lambda row: fetch_owner_from_project_data(row, source="dodge"),
    axis=1
)

NameError: name 'cc_projects' is not defined

In [16]:
def process_one_project_pair(cc_project, dodge_project):
    """
    Process a single pair of projects from ConstructConnect and Dodge.
    Returns the result of the duplicate check.
    """

    cc_project_id = cc_project['ProjectID']
    dodge_project_id = dodge_project['DRNumber']

    try:
        distance, response = is_same_project(
                project_1_data=cc_project,
                project_2_data=dodge_project,
                p1_title_col='Title',
                p2_title_col='ProjectTitle',
                p1_lat_col='Latitude',
                p1_long_col='Longitude',
                p2_lat_col='Lat',
                p2_long_col='Long',
                p1_owner=cc_project['Owner'],
                p2_owner=dodge_project['Owner']
            )
        match = response.project_match.value 
        reasoning = response.Reasoning
        return cc_project_id, dodge_project_id, distance, match, reasoning

    except Exception as e:
        print(f"Error processing project pair: {e}")
        return cc_project_id, dodge_project_id, "error", "error", str(e)

In [28]:
cc_projects_sample = cc_projects.sample(500, random_state=42)
dodge_projects_sample = dodge_projects.sample(500, random_state=42)


In [164]:
from tqdm import tqdm
import concurrent.futures

df_match_results = pd.DataFrame(columns=['cc_project_id', 'dodge_project_id', 'distance', 'match', 'reasoning'])

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = []
    for i, cc_project in cc_projects_sample.iterrows():
        for j, dodge_project in dodge_projects_sample.iterrows():
            futures.append(executor.submit(process_one_project_pair, cc_project, dodge_project))

for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Processing project pairs"):
    cc_project_id, dodge_project_id, distance, match, reasoning = future.result()
    current_match_results = {
        'cc_project_id': cc_project_id,
        'dodge_project_id': dodge_project_id,
        'distance': distance,
        'match': match,
        'reasoning': reasoning
    }

    df_match_results = pd.concat(
        [df_match_results, pd.DataFrame([current_match_results])], 
        ignore_index=True
    )

Processing project pairs:   0%|          | 0/250000 [00:00<?, ?it/s]/var/folders/t9/ynlywvys31d_71jprr25h4c00000gn/T/ipykernel_67703/3250289591.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_match_results = pd.concat(
Processing project pairs: 100%|██████████| 250000/250000 [16:49<00:00, 247.54it/s] 


In [165]:
df_match_results['match'].value_counts()

match
no_match    249999
match            1
Name: count, dtype: int64

In [166]:
df_match_results.to_csv(
    "project_match_results.csv", 
    index=False
)

In [167]:
print(df_match_results[df_match_results['match'] == 'match'].to_markdown())

|       |   cc_project_id |   dodge_project_id |   distance | match   | reasoning                                                                                                                                                                                                                                                                                                                                                                                                                     |
|------:|----------------:|-------------------:|-----------:|:--------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 14782 |      100761670

In [29]:
df_match_results = pd.read_csv("project_match_results.csv")

In [64]:
df_match_results

,cc_project_id,dodge_project_id,distance,match,reasoning
0,1007646137,202500165436,1635.745723,no_match,Distance (miles) between projects exceeds thre...
1,1006839825,202500011097,393.534446,no_match,Distance (miles) between projects exceeds thre...
2,1007196030,202500211789,314.118887,no_match,Distance (miles) between projects exceeds thre...
3,1007480900,202500243187,1074.655178,no_match,Distance (miles) between projects exceeds thre...
4,1005090268,202100705185,2412.473389,no_match,Distance (miles) between projects exceeds thre...
...,...,...,...,...,...
249995,1007653065,202500202522,549.873458,no_match,Distance (miles) between projects exceeds thre...
249996,1007650734,202500243042,720.205763,no_match,Distance (miles) between projects exceeds thre...
249997,1007570059,202500236462,2202.647991,no_match,Distance (miles) between projects exceeds thre...
249998,1007604394,202400470640,1385.370400,no_match,Distance (miles) between projects exceeds thre...


In [50]:
cc_projects_renamed = cc_projects_sample.copy()
dodge_projects_renamed = dodge_projects_sample.copy()

cc_column_mapping = {col: f"cc_{col}" for col in cc_projects_sample.columns}
cc_projects_renamed.rename(columns=cc_column_mapping, inplace=True)
cc_projects_renamed.rename(columns={'cc_ProjectID': 'cc_project_id'}, inplace=True)

dodge_column_mapping = {col: f"dodge_{col}" for col in dodge_projects_sample.columns}
dodge_projects_renamed.rename(columns=dodge_column_mapping, inplace=True)
dodge_projects_renamed.rename(columns={'dodge_DRNumber': 'dodge_project_id'}, inplace=True)


In [51]:
print(cc_column_mapping.keys())
print(dodge_column_mapping.keys())

dict_keys(['ProjectID', 'DataSourceID', 'Title', 'Stage', 'URL', 'UpdateDate', 'IsProspective', 'UpdateText', 'Valuation_Value', 'Valuation_Currency', 'Valuation_ValueType', 'Parameters_Parameter_Ownership', 'Parameters_Parameter_WorkType', 'Parameters_Parameter_CommenceDate', 'Parameters_Parameter_CompletionDate', 'Parameters_Parameter_FloorsAboveGround', 'Parameters_Parameter_Structures', 'Parameters_Parameter_Units', 'Parameters_Parameter_BidDate', 'Parameters_Parameter_BidTime', 'Parameters_Parameter_IsSingleTrade', 'DocumentAvailability_Plans', 'DocumentAvailability_Specs', 'DocumentAvailability_Addenda', 'ParentCategories_PrimaryCategoryName', 'ParentCategories_ParentCategory', 'Addresses_Address', 'Contracts_Contract', 'ProjectEvents_ProjectEvent', 'Companies', 'Users', 'UpdateSummarizations_UpdateSummary', 'PlanSpecs', 'Details_Detail_Scope', 'Details_Detail_Notes', 'Details_Detail', 'RSMeansMaterialDivisions_Division_Concrete', 'RSMeansMaterialDivisions_Division_Metals', 'RSMe

In [65]:

df_match_with_projects = df_match_results.merge(
    cc_projects_renamed,
    left_on='cc_project_id',
    right_on='cc_project_id',
    how='left'
).merge(
    dodge_projects_renamed, 
    left_on='dodge_project_id', 
    right_on='dodge_project_id', 
    how='left')

In [66]:
df_match_with_projects = df_match_with_projects[['cc_project_id', 'cc_Title', 'cc_Details_Detail', 'cc_ParentCategories_ParentCategory',
   'dodge_project_id', 'dodge_ProjectTitle',
    'dodge_FeaturesInfo', 'dodge_PrimaryProjectType', 
    'distance', 'match', 'reasoning']]

In [68]:
df_match_with_projects.to_csv(
    "project_match_results_with_projects.csv", 
    index=False
)

In [67]:
len(df_match_with_projects), len(df_match_with_projects[df_match_with_projects['match'] == 'match'])

(250000, 1)

In [70]:
df_match_results['no-match distance'] = df_match_results.apply(
    lambda row: "Distance (miles) between projects exceeds" in row['reasoning'],
    axis=1
)

In [72]:
df_match_results['no-match distance'].value_counts()

no-match distance
True     249993
False         7
Name: count, dtype: int64

In [77]:
print("Total number of project pairs evaluated:", len(df_match_results))
print("Percent of no-match results due to distance:",
      df_match_results['no-match distance'].sum() / len(df_match_results) * 100)
print("Percent of no-match results due to title similarity (close distance):",
      ((df_match_results['match'] == 'no_match').sum() - df_match_results['no-match distance'].sum()) / len(df_match_results) * 100)
print("Percent of unsure results:",
      (df_match_results['match'] == 'unsure').sum() / len(df_match_results) * 100)
print("Percent of match results:",
        (df_match_results['match'] == 'match').sum() / len(df_match_results) * 100)

Total number of project pairs evaluated: 250000
Percent of no-match results due to distance: 99.99719999999999
Percent of no-match results due to title similarity (close distance): 0.0024000000000000002
Percent of unsure results: 0.0
Percent of match results: 0.00039999999999999996


In [81]:
print(df_match_results[(df_match_results['match'] == 'no_match') & (df_match_results['no-match distance'] == False)].to_markdown())

|        |   cc_project_id |   dodge_project_id |   distance | match    | reasoning                                                                                                                   | no-match distance   |
|-------:|----------------:|-------------------:|-----------:|:---------|:----------------------------------------------------------------------------------------------------------------------------|:--------------------|
|  48451 |      1007626688 |       202500137355 |   0.331064 | no_match | Titles 'conference room 1b av equipment update' and 'job order contracting - area 6' are not similar enough:                | False               |
|        |                 |                    |            |          |                         match ratio 30 < threshold 50                                                                       |                     |
|  51296 |      1007531785 |       202200801982 |   0.319557 | no_match | Titles '2025 pavement marking' and 'sp

## Check Duplication from CRM (FBM)

In [1]:
project_id_duplicates = [1007608639, 1007608638, 1007608637, 1007608636, 1007608634, 1007608633] 

In [19]:
cc_projects_duplicates = get_df_from_bq(
    bq_client_prod, 
    f"""SELECT *, 
    `Addresses_Address`[SAFE_OFFSET(0)].Longitude AS Longitude,
    `Addresses_Address`[SAFE_OFFSET(0)].Latitude AS Latitude,
    FROM `{PROJECT_ID_PROD}.{DATASET_PROD}.{CC_TABLE}` 
    where ProjectID in UNNEST({project_id_duplicates})
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ProjectID ORDER BY sourceFileCreationTime DESC ) = 1"""
)

In [21]:
cc_projects_duplicates['Owner']= cc_projects_duplicates.apply(
    lambda row: fetch_owner_from_project_data(row, source="construct_connect"),
    axis=1
)

In [22]:
from tqdm import tqdm

df_match_results = pd.DataFrame(columns=['cc_project_id_1', 'cc_project_id_2', 
                                         'distance', 'match', 'reasoning'])

for i, project_id_1 in enumerate(project_id_duplicates):
    cc_project_1 = cc_projects_duplicates[cc_projects_duplicates['ProjectID'] == project_id_1].iloc[0]
    
    for j in range(i + 1, len(project_id_duplicates)):
        project_id_2 = project_id_duplicates[j]
        cc_project_2 = cc_projects_duplicates[cc_projects_duplicates['ProjectID'] == project_id_2].iloc[0] 
                
        distance, response = is_same_project(
                        project_1_data=cc_project_1,
                        project_2_data=cc_project_2,
                        p1_title_col='Title',
                        p2_title_col='Title',
                        p1_lat_col='Latitude',
                        p1_long_col='Longitude',
                        p2_lat_col='Latitude',
                        p2_long_col='Longitude',
                        p1_owner=cc_project_1['Owner'],
                        p2_owner=cc_project_2['Owner']
                    )

        match = response.project_match.value 
        reasoning = response.Reasoning
        
        current_match_results = {
            'cc_project_id_1': project_id_1,
            'cc_project_id_2': project_id_2,
            'distance': distance,
            'match': match,
            'reasoning': reasoning
        }
        df_match_results = pd.concat(
            [df_match_results, pd.DataFrame([current_match_results])], 
            ignore_index=True
        )

/var/folders/t9/ynlywvys31d_71jprr25h4c00000gn/T/ipykernel_23122/672222707.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_match_results = pd.concat(


In [24]:
print(df_match_results.to_markdown())

|    |   cc_project_id_1 |   cc_project_id_2 |   distance | match    | reasoning                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
|---:|------------------:|------------------:|-----------:|:---------|:---------------------------------------------------------------------------------------------------------------------------------------------------------